In [4]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. 파일 경로 설정 (내 맥북 로컬 경로)
parquet_path = "volume/shared/2026w09.zstd.parquet"

# 2. Parquet 파일 로드 및 데이터 확인
df = pd.read_parquet(parquet_path)
print(f"Total Rows: {len(df)}")
display(df.head()) # 데이터 구조 미리보기

Total Rows: 9385578


,app_id_hash,country,city,device_board,device_brand,device_carrier_code,device_carrier_name,device_device,device_display_hardware,device_display_height,...,device_iid_hash,device_language,device_locale,device_manufacturer,device_model,device_product,device_sim_code,device_tac,device_total_memory,timestamp
0,bBuoNuXD20vs46Tkej9mvQ==,US,Grand Rapids,G92BoardV1,8849,310260,Tello,TANK2PRO,TANK2PRO_20240411,2460.0,...,Sg8SWTAGriFRTRlntz1PKw==,en,US,Oblue,TANK 2 PRO,TANK2pro,310240,35074458,1.231503e+10,2026-02-23 03:03:39+00:00
1,AXDll7wYh42TsK8Kz6+iJw==,US,Mays Landing,crosshatch,google,310260,Google Fi,crosshatch,SP1A.210812.016.C2,2960.0,...,Th52ttAv/VxE0rPlW0gguQ==,en,US,Google,Pixel 3 XL,crosshatch,310260,99001201,3.753300e+09,2026-02-23 06:37:38+00:00
2,AXDll7wYh42TsK8Kz6+iJw==,US,North Augusta,tokay,google,310260,Google Fi,tokay,BP4A.260205.002,2424.0,...,1R+8y9uKFZGOabNJs3OF0g==,en,US,Google,Pixel 9,tokay,310240,35562419,1.213340e+10,2026-02-23 07:16:00+00:00
3,bBuoNuXD20vs46Tkej9mvQ==,US,Philadelphia,bluejay,google,311480,XFINITY Mobile,bluejay,BP4A.251205.006,2400.0,...,+WT/avXh7qtNARLGNVnAIA==,en,US,Google,Pixel 6a,bluejay,311480,35016289,5.860991e+09,2026-02-23 10:24:43+00:00
4,AXDll7wYh42TsK8Kz6+iJw==,US,Lexington,hiphi,motorola,311480,Visible,hiphi,U1SHS34.1-177-7-3,2400.0,...,UgmqqMQBGvlsZFX6JHvxgw==,en,US,Motorola,motorola edge plus (2022),hiphi_gu,311480,35643969,7.638450e+09,2026-02-23 09:37:33+00:00


In [ ]:
# 3. StarRocks 연결 (MySQL 호환 프로토콜 활용)
SR_USER = "root"
SR_PASS = ""
SR_HOST = "172.16.1.65" 
SR_PORT = "9030" # FE Query Port
SR_DB = "huq"

# 연결 엔진 생성
connection_str = f"mysql+pymysql://{SR_USER}:{SR_PASS}@{SR_HOST}:{SR_PORT}/{SR_DB}"
engine = create_engine(connection_str)

In [ ]:
create_table_sql = """
CREATE TABLE bronze (
    -- Key 컬럼들을 맨 앞으로 배치
    app_id_hash             VARCHAR(255),
    `timestamp`             DATETIME,
    
    -- 나머지 일반 컬럼들
    country                 VARCHAR(10),
    city                    VARCHAR(100),
    device_board            VARCHAR(100),
    device_brand            VARCHAR(100),
    device_carrier_code     VARCHAR(50),
    device_carrier_name     VARCHAR(100),
    device_device           VARCHAR(100),
    device_display_hardware VARCHAR(255),
    device_display_height   VARCHAR(50),
    device_display_width    VARCHAR(50),
    device_fingerprint      VARCHAR(500),
    device_hardware         VARCHAR(100),
    device_host             VARCHAR(100),
    device_iid_hash         VARCHAR(255),
    device_language         VARCHAR(10),
    device_locale           VARCHAR(10),
    device_manufacturer     VARCHAR(100),
    device_model            VARCHAR(100),
    device_product          VARCHAR(100),
    device_sim_code         VARCHAR(50),
    device_tac              VARCHAR(50),
    device_total_memory     VARCHAR(50)
) ENGINE=OLAP
DUPLICATE KEY(app_id_hash, `timestamp`)
DISTRIBUTED BY HASH(app_id_hash) BUCKETS 16;
"""

# DB명을 제외한 기본 연결 주소 (관리용)
ADMIN_CONN = f"mysql+pymysql://{SR_USER}:{SR_PASS}@{SR_HOST}:{SR_PORT}/"
engine_admin = create_engine(ADMIN_CONN)

with engine_admin.begin() as conn:
    # 데이터베이스 생성
    conn.execute(text("CREATE DATABASE IF NOT EXISTS huq"))
    print("✅ 데이터베이스 'huq' 준비 완료")

with engine.begin() as conn:
    # 기존 테이블이 있다면 삭제 (if_exists='replace' 효과)
    conn.execute(text("DROP TABLE IF EXISTS bronze"))
    
    # 새 테이블 생성
    conn.execute(text(create_table_sql))
    print("✅ 테이블 'bronze' 생성 완료")

✅ 데이터베이스 'huq' 준비 완료
✅ 테이블 'bronze' 생성 완료


In [ ]:
import requests
import io

# 1. 설정 (기존 설정 활용)
SR_HOST = "172.16.1.65"
SR_FE_HTTP_PORT = "8030"  # 주의: Query Port(9030)가 아닌 HTTP Port(8030)입니다.
SR_DB = "huq"
SR_TABLE = "bronze"
SR_USER = "root"
SR_PASS = ""

# 1. 설정 (8030 포트를 사용해야 합니다)
SR_HTTP_PORT = "8030" 
url = f"http://{SR_HOST}:{SR_HTTP_PORT}/api/{SR_DB}/{SR_TABLE}/_stream_load"

cols = [
    'app_id_hash', 'timestamp', 'country', 'city', 'device_board', 'device_brand', 
    'device_carrier_code', 'device_carrier_name', 'device_device', 'device_display_hardware', 
    'device_display_height', 'device_display_width', 'device_fingerprint', 'device_hardware', 
    'device_host', 'device_iid_hash', 'device_language', 'device_locale', 'device_manufacturer', 
    'device_model', 'device_product', 'device_sim_code', 'device_tac', 'device_total_memory'
]

# 2. 전송 함수 정의
def stream_load_chunk(chunk_df, chunk_num):
    # 메모리 내에서 CSV(Tab 구분자) 형태로 변환
    buffer = io.StringIO()
    chunk_df.to_csv(buffer, index=False, header=False, sep='\t')
    
    headers = {
        "Expect": "100-continue",
        "label": f"load_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}_{chunk_num}",
        "column_separator": "\\t", # 역슬래시 두 번 (탭 구분자)
        "columns": ",".join(cols),
        "format": "csv",
        "Discovery": "false"
    }
    
    auth = (SR_USER, SR_PASS)
    response = requests.put(
        url, 
        data=buffer.getvalue().encode('utf-8'), 
        headers=headers, 
        auth=auth,
        allow_redirects=False
    )
    return response.json()

# 3. 분할 실행 (10만 건씩)
CHUNK_SIZE = 100000 
total_rows = len(df)

print(f"🚀 Stream Loading 시작: {total_rows} rows...")

for i in range(0, total_rows, CHUNK_SIZE):
    chunk = df.iloc[i : i + CHUNK_SIZE]
    chunk_num = (i // CHUNK_SIZE) + 1
    
    print(f"📦 [{chunk_num}] {i} ~ {min(i + CHUNK_SIZE, total_rows)} 전송 중...", end=" ")
    
    result = stream_load_chunk(chunk, chunk_num)
    
    if result.get("Status") == "Success":
        print(f"✅ 완료 ({result.get('LoadTimeMs')}ms)")
    else:
        print(f"❌ 실패: {result.get('Message')}")
        # 상세 에러가 있는 URL 출력 (브라우저에서 확인 가능)
        if result.get("ErrorURL"):
            print(f"   👉 에러 확인: {result.get('ErrorURL')}")
        break

print("\n✨ 모든 작업이 완료되었습니다.")

🚀 Stream Loading 시작: 9385578 rows...
📦 [1] 0 ~ 100000 전송 중... 

ConnectionError: HTTPConnectionPool(host='kube-starrocks-be-1.kube-starrocks-be-search.starrocks.svc.cluster.local', port=8040): Max retries exceeded with url: /api/huq/bronze/_stream_load (Caused by NameResolutionError("HTTPConnection(host='kube-starrocks-be-1.kube-starrocks-be-search.starrocks.svc.cluster.local', port=8040): Failed to resolve 'kube-starrocks-be-1.kube-starrocks-be-search.starrocks.svc.cluster.local' ([Errno 8] nodename nor servname provided, or not known)"))